# 00 · Build the analysis panel

This notebook builds the **ticker-day panel** used by the return and order-flow notebooks.

**What it does**

1. Load the licensed data files (TAQ, CRSP, earnings, Quiver).
2. Merge CRSP returns and apply the sample filters.
3. Add the control variables and order-flow measures.
4. Save the panel to `data/licensed/work_panel.parquet`.

> All the heavy work lives in `helpers.build_panel()`, so this notebook stays short.
> Run this notebook **first**: the other notebooks read the panel it saves.

In [1]:
import sys
from pathlib import Path

# Make the repo root importable so we can use config.py and helpers.py.
here = Path.cwd()
ROOT = here if (here / "helpers.py").exists() else here.parent
sys.path.insert(0, str(ROOT))

from config import LICENSED_DATA, TABLE_DIR
from helpers import build_panel

## Build the panel

This reads the big CSV files, so it can take a minute or two.

In [2]:
panel, sample_table, senate_table = build_panel()

print("Panel rows:", len(panel))
print("Tickers:", panel["ticker"].nunique())
print("Date range:", panel["date"].min().date(), "to", panel["date"].max().date())

Panel rows: 342016
Tickers: 1402
Date range: 2018-10-01 to 2024-12-31


## Sample construction

How the raw Quiver disclosures shrink down to the final panel (matches Table: Sample construction).

In [3]:
sample_table

,step,observations,tickers
0,"Raw Quiver disclosures (all dates, all asset t...",110024,5068
1,Filed 2018-11-05 to 2024-12-31,60092,3561
2,Qualifying purchase rows after filters,13588,2115
3,Ticker-day cells after mapping Filed to next s...,10542,2115
4,Cells that intersect the raw TAQ panel,9693,1784
5,Raw TAQ panel ticker-days,438770,2138
6,Final filtered TAQ/CRSP panel ticker-days,342016,1402
7,Of which treated purchase ticker-days,7509,1165


## Senate event sample

The Senate purchases that drive the main analysis (matches Table: Senate event sample).

In [4]:
senate_table

,step,observations,tickers
0,Qualifying purchase rows after filters,13588,2115
1,Senate purchase rows within this universe,1939,580
2,Senate ticker-event cells after filing-date ma...,1419,580
3,Senate treated ticker-days in the final panel,1080,382
4,Final filtered TAQ/CRSP panel ticker-days,342016,1402


## Save

Save the panel and the two tables. The panel is large and licensed, so it is **not** committed to git.

In [5]:
panel.to_parquet(LICENSED_DATA / "work_panel.parquet", index=False)
sample_table.to_csv(TABLE_DIR / "sample_construction.csv", index=False)
senate_table.to_csv(TABLE_DIR / "senate_sample_counts.csv", index=False)

print("Saved work_panel.parquet and the two sample tables.")

Saved work_panel.parquet and the two sample tables.
